# Runtime-Only Discrete QSL Check

This notebook checks only

$$
S_N = \sum_{k=1}^{N}\frac{2\,\Delta E_k}{\hbar}\,\delta t \ge S_0,
$$

with a single runtime-based step definition

$$
\delta t = \frac{T_{\mathrm{VQE}}}{N_{\mathrm{iter}}}.
$$

Here,

$$
\Delta E_k = \sqrt{\langle H^2\rangle_k - \langle H\rangle_k^2},
\qquad
S_0 = \arccos\!\left(\left|\langle\psi_0\mid\psi_f\rangle\right|\right).
$$


In [5]:
import time
import numpy as np
from scipy.optimize import minimize

from qiskit import QuantumCircuit
from qiskit.circuit import Parameter
from qiskit.quantum_info import SparsePauliOp, Statevector

hbar = 1.0

# Ansatz: CNOT(Rz(t2)_1 Ry(t1)_0 |00>)
t1 = Parameter("t1")
t2 = Parameter("t2")
ansatz = QuantumCircuit(2)
ansatz.ry(t1, 0)
ansatz.rz(t2, 1)
ansatz.cx(0, 1)

# Hamiltonian: H = X \otimes X
H = SparsePauliOp.from_list([("XX", 1.0)])
H2 = (H @ H).simplify()


In [6]:
def state_from_params(params):
    bound = ansatz.assign_parameters({t1: float(params[0]), t2: float(params[1])}, inplace=False)
    return Statevector.from_instruction(bound)


def energy_delta_state(params):
    psi = state_from_params(params)
    e = float(np.real(psi.expectation_value(H)))
    e2 = float(np.real(psi.expectation_value(H2)))
    var = max(0.0, e2 - e**2)
    delta_e = float(np.sqrt(var))
    return e, delta_e, psi


history = {
    "params": [],
    "energy": [],
    "deltaE": [],
    "state": [],
}


def record_point(params):
    params = np.asarray(params, dtype=float)
    if history["params"] and np.linalg.norm(params - history["params"][-1]) < 1e-12:
        return
    e, delta_e, psi = energy_delta_state(params)
    history["params"].append(params.copy())
    history["energy"].append(e)
    history["deltaE"].append(delta_e)
    history["state"].append(psi)


init_params = np.array([0.0, 0.0], dtype=float)

vqe_start = time.perf_counter()
record_point(init_params)


def objective(params):
    e, _, _ = energy_delta_state(params)
    return e


def callback(xk):
    record_point(xk)


result = minimize(
    objective,
    init_params,
    method="COBYLA",
    callback=callback,
    options={"maxiter": 120, "rhobeg": 0.5, "tol": 1e-4},
)

record_point(result.x)
vqe_total_runtime = time.perf_counter() - vqe_start

energies = np.asarray(history["energy"])
deltaE = np.asarray(history["deltaE"])
states = history["state"]


In [7]:
if len(states) < 2:
    raise RuntimeError("Need at least one optimization step to evaluate the runtime-based QSL sum.")

n_steps = len(states) - 1
step_idx = np.arange(1, n_steps + 1)

# Runtime-only step definition.
delta_t = float(vqe_total_runtime / n_steps)

# v_k = 2 DeltaE_k / hbar, using the state at step k.
speed = 2.0 * deltaE[1:] / hbar
terms = speed * delta_t
S_N = np.cumsum(terms)

ov_0f = np.clip(abs(np.vdot(states[0].data, states[-1].data)), 0.0, 1.0)
S0 = float(np.arccos(ov_0f))

final_relation_holds = bool(S_N[-1] + 1e-10 >= S0)
cross = np.where(S_N >= S0 - 1e-10)[0]
N_min = int(cross[0] + 1) if cross.size else None


In [8]:
print("Runtime-only QSL summary")
print("-" * 72)
print(f"Optimization success                             : {result.success}")
print(f"Termination message                              : {result.message}")
print(f"Total VQE runtime [s]                            : {vqe_total_runtime:.10f}")
print(f"Total iterations N_iter                          : {n_steps}")
print(f"delta t = T_VQE / N_iter [s]                     : {delta_t:.10f}")
print(f"S0 (initial to final geodesic angle)             : {S0:.10f}")
print(f"S_N final (runtime-only)                         : {S_N[-1]:.10f}")
print(f"Relation S_N >= S0 at final iterate              : {final_relation_holds}")
print(f"Smallest N with S_N >= S0                        : {N_min}")


Runtime-only QSL summary
------------------------------------------------------------------------
Optimization success                             : True
Termination message                              : Return from COBYLA because the trust region radius reaches its lower bound.
Total VQE runtime [s]                            : 0.8980011884
Total iterations N_iter                          : 8
delta t = T_VQE / N_iter [s]                     : 0.1122501486
S0 (initial to final geodesic angle)             : 0.7853598238
S_N final (runtime-only)                         : 0.2248483174
Relation S_N >= S0 at final iterate              : False
Smallest N with S_N >= S0                        : None


## Note

This notebook intentionally keeps only the runtime-based check with
$\delta t = T_{\mathrm{VQE}} / N_{\mathrm{iter}}$.
